# Sign

$$
\text{Sign}(x) \;=\; \text{Sign}_{0,1,0}(x) \;=\; \begin{cases} 1 & x > 0 \\ 0 & x = 0 \\ -1 & x < 0 \end{cases}
$$


$$
\text{Sign}_{a,b,c}(x) \;=\; a + (b-a)\,\text{Sign}(x - c) \;=\; \begin{cases} b & x > c \\ \dfrac{a+b}{2} & x = c \\ a & x < c \end{cases}
$$

$$
\begin{aligned}
(x == a) &:= (x > a - \epsilon) \cdot (x < a + \epsilon) \\
(x_i == 0) &:= (x_i \geq -0.01) \cdot (x_i \leq 0.01) \\[6pt]
x_i = 0.005: \quad &(0.005 \geq -0.01) \cdot (0.005 \leq 0.01) = 1 \cdot 1 = 1 \\
x_i = 0.5: \quad &(0.5 \geq -0.01) \cdot (0.5 \leq 0.01) = 1 \cdot 0 = 0
\end{aligned}
$$

### Sigmoid Convergence

#### Plotted Approximation Dataset Output

In [1]:
import pandas as pd
import plotly.express as px

# 1. Load the data
df = pd.read_csv("sigmoid_convergence_results.csv")

# Clean up the column names by removing '#' and any extra spaces:
df.columns = df.columns.str.replace('#', '').str.strip()

df = df.rename(columns={
    "sigmoid_approximation": "Sigmoid Approximation",
    "sign_real": "Sign Real"
})

# 2. Filter for rows where sigmoid_approximation is within range
filtered_df = df[
    (df['Sigmoid Approximation'] >= -0.5) & 
    (df['Sigmoid Approximation'] <= 1.5) &
    (df['x_value'] >= -0.03) &   # ← also filter x, not just y
    (df['x_value'] <= 0.03)
]
fig = px.line(
    filtered_df, # use the unfiltered df
    x="x_value", 
    y=["Sigmoid Approximation", "Sign Real"],
    title="Sigmoid Approximation terms = 10 , k=32 and Sign Real",
    labels={
        "x_value": "Plaintext Input", 
        "value": "Calculated Value", 
        "variable": "Metrics",
    }
)

# 2. Hardcode the window for BOTH axes so they are identical
fig.update_xaxes(range=[-0.03, 0.03])
fig.update_yaxes(range=[-0.5, 1.5])

fig.show()

### Convergence Analysis 


In [46]:
import pandas as pd
import numpy as np

df = pd.read_csv("sigmoid_convergence_results.csv")

valid_df = df[
    (df['sigmoid_approximation'] >= -0.5) &
    (df['sigmoid_approximation'] <= 1.5)
].copy().sort_values('x_value').reset_index(drop=True)

def empirical_convergence_report(df):
    left_df  = df[df['x_value'] < 0].copy()
    right_df = df[df['x_value'] > 0].copy()

    # --- find empirical convergence radii via local extrema ---
    rhs_peak_idx = right_df['sigmoid_approximation'].idxmax()
    rhs_peak_x   = right_df.loc[rhs_peak_idx, 'x_value']
    rhs_peak_val = right_df.loc[rhs_peak_idx, 'sigmoid_approximation']

    lhs_trough_idx = left_df['sigmoid_approximation'].idxmin()
    lhs_trough_x   = left_df.loc[lhs_trough_idx, 'x_value']
    lhs_trough_val = left_df.loc[lhs_trough_idx, 'sigmoid_approximation']

    # --- compute % off within each convergence window ---
    rhs_window = right_df[right_df['x_value'] <= rhs_peak_x]
    lhs_window = left_df[left_df['x_value']  >= lhs_trough_x]

    # RHS: % off from 1
    rhs_pct_off = (1 - rhs_window['sigmoid_approximation']).abs()
    rhs_max_pct_off  = rhs_pct_off.max() * 100
    rhs_mean_pct_off = rhs_pct_off.mean() * 100

    # LHS: % off from 0
    lhs_pct_off = lhs_window['sigmoid_approximation'].abs()
    lhs_max_pct_off  = lhs_pct_off.max() * 100
    lhs_mean_pct_off = lhs_pct_off.mean() * 100

    # --- avg % off excluding 0 ---
    rhs_excl_zero = rhs_window[rhs_window['x_value'] > 0]
    rhs_avg_pct_off = (1 - rhs_excl_zero['sigmoid_approximation']).abs().mean() * 100

    lhs_excl_zero = lhs_window[lhs_window['x_value'] < 0]
    lhs_avg_pct_off = lhs_excl_zero['sigmoid_approximation'].abs().mean() * 100

    # --- report ---
    print("Empirical Convergence Radius Analysis")
    print("=" * 60)
    print(f"\nRHS convergence radius (local max):  x = {rhs_peak_x:.4f}")
    print(f"  Peak value:                          {rhs_peak_val:.6f}  (ideal = 1.0)")
    print(f"  Peak % off from 1:                   {abs(1 - rhs_peak_val)*100:.2f}%")
    print(f"  Avg % off from 1 on (0, {rhs_peak_x:.4f}):      {rhs_avg_pct_off:.2f}%")

    print(f"\nLHS convergence radius (local min):  x = {lhs_trough_x:.4f}")
    print(f"  Trough value:                        {lhs_trough_val:.6f}  (ideal = 0.0)")
    print(f"  Trough % off from 0:                 {abs(lhs_trough_val)*100:.2f}%")
    print(f"  Avg % off from 0 on ({lhs_trough_x:.4f}, 0):   {lhs_avg_pct_off:.2f}%")

    print()
    print("Printable statements:")
    print("─" * 60)
    print(f"  On (0, {rhs_peak_x:.4f}), the approximation is at most "
          f"{rhs_max_pct_off:.2f}% away from 1 (mean: {rhs_avg_pct_off:.2f}%).")
    print(f"  On ({lhs_trough_x:.4f}, 0), the approximation is at most "
          f"{lhs_max_pct_off:.2f}% away from 0 (mean: {lhs_avg_pct_off:.2f}%).")

    return rhs_peak_x, lhs_trough_x

empirical_convergence_report(valid_df)

Empirical Convergence Radius Analysis

RHS convergence radius (local max):  x = 0.0200
  Peak value:                          0.916460  (ideal = 1.0)
  Peak % off from 1:                   8.35%
  Avg % off from 1 on (0, 0.0200):      23.21%

LHS convergence radius (local min):  x = -0.0200
  Trough value:                        0.083540  (ideal = 0.0)
  Trough % off from 0:                 8.35%
  Avg % off from 0 on (-0.0200, 0):   23.21%

Printable statements:
────────────────────────────────────────────────────────────
  On (0, 0.0200), the approximation is at most 46.80% away from 1 (mean: 23.21%).
  On (-0.0200, 0), the approximation is at most 46.80% away from 0 (mean: 23.21%).


(np.float64(0.02), np.float64(-0.02))

### Updated Convergence Anaylsis 

In [4]:
# Per-k accuracy of the sigmoid sign approximation vs. the true sign function.

import pandas as pd
import numpy as np

df = pd.read_csv("tests/sigmoid_convergence_results.csv")

# Error tolerance for the "accurate" sub-window. The approximation's accuracy
# ceiling is ~8.3% (its best value is 0.917, never reaching 1), so a 10% band
# is just wide enough to admit a usable window near the convergence radius.
ACC_TOL = 0.10


def accuracy_within_window(sub_k):
    """For a single k: locate the empirical convergence window, measure how far
    the approximation is from the true sign inside it, and find the sub-range
    on each side where that error stays within ACC_TOL.

    The window edge is the curve's turning point, searched only within the
    theoretical radius pi/k where the sigmoid Taylor series converges -- so no
    arbitrary value mask is needed. (Sigmoid's nearest complex singularities sit
    at +/-i*pi, giving radius pi/k for the k-scaled input; tanh's are at
    +/-i*pi/2, hence the narrower pi/(2k) used in the tanh notebook.) Error is
    |approx - sign_real| (distance from 1 for x>0, from 0 for x<0)."""
    k = int(sub_k['k'].iloc[0])
    radius = np.pi / k

    left  = sub_k[(sub_k['x_value'] < 0) & (sub_k['x_value'] >= -radius)]
    right = sub_k[(sub_k['x_value'] > 0) & (sub_k['x_value'] <=  radius)]

    # Empirical radius on each side = the turning point inside the convergent zone.
    rhs_radius = right.loc[right['sigmoid_approximation'].idxmax(), 'x_value']
    lhs_radius = left.loc[left['sigmoid_approximation'].idxmin(),  'x_value']

    rhs_win = right[right['x_value'] <= rhs_radius].copy()
    lhs_win = left[left['x_value']  >= lhs_radius].copy()

    rhs_win['err'] = (rhs_win['sigmoid_approximation'] - rhs_win['sign_real']).abs()
    lhs_win['err'] = (lhs_win['sigmoid_approximation'] - lhs_win['sign_real']).abs()

    # Sub-range on each side where error <= ACC_TOL (empty -> NaN bounds).
    rhs_ok = rhs_win.loc[rhs_win['err'] <= ACC_TOL, 'x_value']
    lhs_ok = lhs_win.loc[lhs_win['err'] <= ACC_TOL, 'x_value']
    rhs_lo, rhs_hi = (rhs_ok.min(), rhs_ok.max()) if len(rhs_ok) else (np.nan, np.nan)
    lhs_lo, lhs_hi = (lhs_ok.min(), lhs_ok.max()) if len(lhs_ok) else (np.nan, np.nan)

    return {
        'k': k,
        'lhs_radius':  lhs_radius,
        'rhs_radius':  rhs_radius,
        'lhs_max_%':   lhs_win['err'].max()  * 100,
        'rhs_max_%':   rhs_win['err'].max()  * 100,
        # range where error stays within ACC_TOL
        'lhs_<=tol':   (lhs_lo, lhs_hi),
        'rhs_<=tol':   (rhs_lo, rhs_hi),
    }


rows   = [accuracy_within_window(g) for _, g in df.groupby('k')]
report = pd.DataFrame(rows).sort_values('k').reset_index(drop=True)

pd.set_option('display.float_format', lambda v: f"{v:.3f}")
print(f"Sigmoid sign accuracy vs. true sign, per k   (error % = 100 * |approx - sign|)")
print(f"Within-tolerance ranges use ACC_TOL = {ACC_TOL:.0%}")
print("=" * 78)
print(report.to_string(index=False))

print("\nPrintable statements:")
print("-" * 78)
for r in rows:
    (llo, lhi), (rlo, rhi) = r['lhs_<=tol'], r['rhs_<=tol']
    print(f"  k={r['k']:>2}: within {ACC_TOL:.0%} of true sign on "
          f"x in [{llo:.3f}, {lhi:.3f}] (left) and "
          f"[{rlo:.3f}, {rhi:.3f}] (right).")


Sigmoid sign accuracy vs. true sign, per k   (error % = 100 * |approx - sign|)
Within-tolerance ranges use ACC_TOL = 10%
 k  lhs_radius  rhs_radius  lhs_max_%  rhs_max_%        lhs_<=tol      rhs_<=tol
 1      -2.543       2.543     49.975     49.975  (-2.543, -2.21)  (2.21, 2.543)
 2      -1.272       1.272     49.950     49.950 (-1.272, -1.105) (1.105, 1.272)
 4      -0.636       0.636     49.900     49.900 (-0.636, -0.553) (0.553, 0.636)
 8      -0.318       0.318     49.800     49.800 (-0.318, -0.277) (0.277, 0.318)
16      -0.159       0.159     49.600     49.600 (-0.159, -0.139) (0.139, 0.159)
32      -0.079       0.079     49.200     49.200  (-0.079, -0.07)  (0.07, 0.079)
64      -0.040       0.040     48.401     48.401  (-0.04, -0.035)  (0.035, 0.04)

Printable statements:
------------------------------------------------------------------------------
  k= 1: within 10% of true sign on x in [-2.543, -2.210] (left) and [2.210, 2.543] (right).
  k= 2: within 10% of true sign on x 

In [3]:
# Visualize the k=1 sigmoid sign approximation with its <=10% accuracy ring shaded.

import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_csv("tests/sigmoid_convergence_results.csv")
ACC_TOL = 0.10


def accuracy_within_window(sub_k):
    k = int(sub_k['k'].iloc[0])
    radius = np.pi / k

    left  = sub_k[(sub_k['x_value'] < 0) & (sub_k['x_value'] >= -radius)]
    right = sub_k[(sub_k['x_value'] > 0) & (sub_k['x_value'] <=  radius)]

    rhs_radius = right.loc[right['sigmoid_approximation'].idxmax(), 'x_value']
    lhs_radius = left.loc[left['sigmoid_approximation'].idxmin(),  'x_value']

    rhs_win = right[right['x_value'] <= rhs_radius].copy()
    lhs_win = left[left['x_value']  >= lhs_radius].copy()

    rhs_win['err'] = (rhs_win['sigmoid_approximation'] - rhs_win['sign_real']).abs()
    lhs_win['err'] = (lhs_win['sigmoid_approximation'] - lhs_win['sign_real']).abs()

    rhs_ok = rhs_win.loc[rhs_win['err'] <= ACC_TOL, 'x_value']
    lhs_ok = lhs_win.loc[lhs_win['err'] <= ACC_TOL, 'x_value']
    rhs_lo, rhs_hi = (rhs_ok.min(), rhs_ok.max()) if len(rhs_ok) else (np.nan, np.nan)
    lhs_lo, lhs_hi = (lhs_ok.min(), lhs_ok.max()) if len(lhs_ok) else (np.nan, np.nan)

    return {
        'k': k,
        'lhs_<=tol': (lhs_lo, lhs_hi),
        'rhs_<=tol': (rhs_lo, rhs_hi),
    }


K = 1
sub = df[df['k'] == K].sort_values('x_value')
stats = accuracy_within_window(sub)
(llo, lhi) = stats['lhs_<=tol']
(rlo, rhi) = stats['rhs_<=tol']

# Mask the divergent Taylor tail — sigmoid lives in [0,1], clip outside [-0.5, 1.5].
plot = sub.copy()
plot['sigmoid_approximation'] = plot['sigmoid_approximation'].where(
    plot['sigmoid_approximation'].between(-0.5, 1.5), np.nan
)

fig = go.Figure()

# Shaded <=10% accuracy rings (left and right), drawn first so curves sit on top.
for lo, hi in [(llo, lhi), (rlo, rhi)]:
    fig.add_vrect(
        x0=lo, x1=hi,
        fillcolor="green", opacity=0.18, line_width=0,
        annotation_text=f"≤{ACC_TOL:.0%} error",
        annotation_position="top left",
    )

fig.add_trace(go.Scatter(
    x=plot['x_value'], y=plot['sigmoid_approximation'],
    mode="lines", name=f"Sigmoid approx (k={K}, n_terms=9)",
    line=dict(color="royalblue", width=2),
))
fig.add_trace(go.Scatter(
    x=sub['x_value'], y=sub['sign_real'],
    mode="lines", name="True Sign",
    line=dict(color="black", dash="dash", width=2),
))

fig.update_xaxes(title="Plaintext Input", range=[-4, 4])
fig.update_yaxes(title="Calculated Value", range=[-0.5, 1.5])
fig.update_layout(
    title=f"Sigmoid Sign Approximation (k={K}) with ≤{ACC_TOL:.0%} Accuracy Ring Shaded",
)
fig.show()


#### Presentation Sigmoid Convergence Analysis

In [ ]:
import numpy as np
import plotly.graph_objects as go

def exact_sigmoid(x):
    return 1 / (1 + np.exp(-x))

def approx_sigmoid10(x):
    coeffs = [
        (0,  1,        2),
        (1,  1,        4),
        (3, -1,        48),
        (5,  1,        480),
        (7, -17,       80640),
        (9,  31,       1451520),
        (11, -691,     319334400),
        (13,  5461,    24908083200),
        (15, -929569,  429079654400),
    ]
    result = np.zeros_like(x, dtype=float)
    for deg, num, den in coeffs:
        result += (num / den) * np.power(x, deg)
    return result

xs = np.linspace(-5.0, 5.0, 800)
exact  = exact_sigmoid(xs)
approx = approx_sigmoid10(xs)

PRACTICAL_BOUND = 2.0
CLIP = 1.5
approx_clipped = np.where((np.abs(approx) > CLIP) | (approx < -0.1), np.nan, approx)

fig = go.Figure()

# divergence shading
fig.add_vrect(
    x0=-5.0, x1=-PRACTICAL_BOUND,
    fillcolor="rgba(220,50,50,0.08)",
    layer="below", line_width=0,
    annotation_text="diverges", annotation_position="top left",
    annotation_font_size=11, annotation_font_color="rgba(180,40,40,0.7)"
)
fig.add_vrect(
    x0=PRACTICAL_BOUND, x1=5.0,
    fillcolor="rgba(220,50,50,0.08)",
    layer="below", line_width=0,
    annotation_text="diverges", annotation_position="top right",
    annotation_font_size=11, annotation_font_color="rgba(180,40,40,0.7)"
)

# practical boundary lines
for xv in [-PRACTICAL_BOUND, PRACTICAL_BOUND]:
    fig.add_vline(
        x=xv,
        line_dash="dash", line_color="rgba(220,50,50,0.45)", line_width=1.2,
    )

# exact sigmoid
fig.add_trace(go.Scatter(
    x=xs, y=exact,
    mode="lines",
    name="exact sigmoid(x)",
    line=dict(color="#534AB7", width=2.5),
))

# 10-term approximation
fig.add_trace(go.Scatter(
    x=xs, y=approx_clipped,
    mode="lines",
    name="10-term Taylor approx",
    line=dict(color="#0F6E56", width=2, dash="dash"),
    connectgaps=False,
))

# boundary labels
for xv, label in [(-PRACTICAL_BOUND, "-2"), (PRACTICAL_BOUND, "2")]:
    fig.add_annotation(
        x=xv, y=1.42,
        text=label,
        showarrow=False,
        font=dict(size=12, color="rgba(180,40,40,0.8)"),
    )

fig.add_annotation(
    x=0, y=-0.12,
    text="practical bound |x| < 2  (theoretical radius |x| < π with infinite terms)",
    showarrow=False,
    font=dict(size=11, color="rgba(100,100,100,0.85)"),
    bgcolor="rgba(255,255,255,0.6)",
)

fig.update_layout(
    title=dict(
        text="sigmoid(x): exact vs 10-term Taylor approx  (practical convergence |x| < 2)",
        font=dict(size=15),
    ),
    xaxis=dict(
        title="x",
        range=[-5.0, 5.0],
        zeroline=True, zerolinewidth=1, zerolinecolor="rgba(150,150,150,0.4)",
        gridcolor="rgba(150,150,150,0.15)",
    ),
    yaxis=dict(
        title="sigmoid(x)",
        range=[-0.1, 1.5],
        zeroline=True, zerolinewidth=1, zerolinecolor="rgba(150,150,150,0.4)",
        gridcolor="rgba(150,150,150,0.15)",
    ),
    legend=dict(x=0.02, y=0.97, bgcolor="rgba(255,255,255,0.7)", borderwidth=0),
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=750,
    height=420,
    margin=dict(l=60, r=40, t=50, b=50),
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

def exact_sigmoid(x):
    return 1 / (1 + np.exp(-x))

def approx_sigmoid10(x):
    coeffs = [
        (0,  1,        2),
        (1,  1,        4),
        (3, -1,        48),
        (5,  1,        480),
        (7, -17,       80640),
        (9,  31,       1451520),
        (11, -691,     319334400),
        (13,  5461,    24908083200),
        (15, -929569,  429079654400),
    ]
    result = np.zeros_like(x, dtype=float)
    for deg, num, den in coeffs:
        result += (num / den) * np.power(x, deg)
    return result

xs = np.linspace(-5.0, 5.0, 800)
exact  = exact_sigmoid(xs)
approx = approx_sigmoid10(xs)

PRACTICAL_BOUND = 2.0
CLIP = 1.5
approx_clipped = np.where((np.abs(approx) > CLIP) | (approx < -0.1), np.nan, approx)

fig = go.Figure()

# divergence shading
fig.add_vrect(
    x0=-5.0, x1=-PRACTICAL_BOUND,
    fillcolor="rgba(220,50,50,0.08)",
    layer="below", line_width=0,
    annotation_text="diverges", annotation_position="top left",
    annotation_font_size=11, annotation_font_color="rgba(180,40,40,0.7)"
)
fig.add_vrect(
    x0=PRACTICAL_BOUND, x1=5.0,
    fillcolor="rgba(220,50,50,0.08)",
    layer="below", line_width=0,
    annotation_text="diverges", annotation_position="top right",
    annotation_font_size=11, annotation_font_color="rgba(180,40,40,0.7)"
)

# practical boundary lines
for xv in [-PRACTICAL_BOUND, PRACTICAL_BOUND]:
    fig.add_vline(
        x=xv,
        line_dash="dash", line_color="rgba(220,50,50,0.45)", line_width=1.2,
    )

# exact sigmoid
fig.add_trace(go.Scatter(
    x=xs, y=exact,
    mode="lines",
    name="exact sigmoid(x)",
    line=dict(color="#534AB7", width=2.5),
))

# 10-term approximation
fig.add_trace(go.Scatter(
    x=xs, y=approx_clipped,
    mode="lines",
    name="10-term Taylor approx",
    line=dict(color="#0F6E56", width=2, dash="dash"),
    connectgaps=False,
))

# boundary labels
for xv, label in [(-PRACTICAL_BOUND, "-2"), (PRACTICAL_BOUND, "2")]:
    fig.add_annotation(
        x=xv, y=1.42,
        text=label,
        showarrow=False,
        font=dict(size=12, color="rgba(180,40,40,0.8)"),
    )

fig.add_annotation(
    x=0, y=-0.12,
    text="practical bound |x| < 2  (theoretical radius |x| < π with infinite terms)",
    showarrow=False,
    font=dict(size=11, color="rgba(100,100,100,0.85)"),
    bgcolor="rgba(255,255,255,0.6)",
)

fig.update_layout(
    title=dict(
        text="sigmoid(x): exact vs 10-term Taylor approx  (practical convergence |x| < 2)",
        font=dict(size=15),
    ),
    xaxis=dict(
        title="x",
        range=[-5.0, 5.0],
        zeroline=True, zerolinewidth=1, zerolinecolor="rgba(150,150,150,0.4)",
        gridcolor="rgba(150,150,150,0.15)",
    ),
    yaxis=dict(
        title="sigmoid(x)",
        range=[-0.1, 1.5],
        zeroline=True, zerolinewidth=1, zerolinecolor="rgba(150,150,150,0.4)",
        gridcolor="rgba(150,150,150,0.15)",
    ),
    legend=dict(x=0.02, y=0.97, bgcolor="rgba(255,255,255,0.7)", borderwidth=0),
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=750,
    height=420,
    margin=dict(l=60, r=40, t=50, b=50),
)

fig.show()

$$
\begin{aligned}
\operatorname{Sigmoid}(x)
&= \frac{1}{1+e^{-x}} \\[6pt]
&= \frac{1}{1+\left(1 - x + \frac{x^2}{2!} - \frac{x^3}{3!} + \cdots \right)} \\[6pt]
&= \frac{1}{1 - u},
\end{aligned}
$$

where

$$
u = -1 + x - \frac{x^2}{2!} + \frac{x^3}{3!} - \cdots
$$

and therefore

$$
\frac{1}{1 - u} = 1 + u + u^2 + u^3 + \cdots.
$$

$$
\sigma(x) \approx \frac{1}{2} + \frac{1}{4}x - \frac{1}{48}x^3 + \frac{1}{480}x^5 - \frac{17}{80640}x^7 + \frac{31}{1451520}x^9 - \frac{691}{319334400}x^{11} + \frac{5461}{24908083200}x^{13} - \frac{929569}{429079654400}x^{15}
$$

Sigmoid n terms converges for $|x| < \pi$, practical 10 terms: $|x| < 2$




$$
\sigma(x) \approx \sum_{k=0}^{n} \frac{(-1)^k B_{2k}}{(2k)!} \cdot 4^k (4^k - 1) \cdot x^{2k-1} + \frac{1}{2}
$$

where $B_{2k}$ are Bernoulli numbers. Converges for $|x| < \pi$.



$$
\sigma(x) = \frac{1 + \tanh\left(\frac{x}{2}\right)}{2}
$$


$$
\sigma(x) \;=\; \frac{1 + \tanh\left(\frac{x}{2}\right)}{2} \;\approx\; \sum_{k=0}^{n} \frac{(-1)^k B_{2k}}{(2k)!} \cdot 4^k (4^k - 1) \cdot x^{2k-1} + \frac{1}{2}
$$

where $B_{2k}$ are Bernoulli numbers. Converges for $|x| < \pi$.

### Comparison

In [ ]:
import plotly.graph_objects as go

# Data
n_terms = [1, 3, 5, 7, 9]

sig_adds = [2, 6, 10, 14, 18]
sig_mults = [3, 15, 31, 51, 71]
sig_levels = [13, 9, 7, 7, 5]

tanh_adds = [0, 2, 4, 6, 8]
tanh_mults = [1, 10, 22, 36, 51]
tanh_levels = [14, 11, 10, 10, 9]

fig = go.Figure()

# Sigmoid
fig.add_trace(go.Bar(
    x=[f"{n}<br>Sigmoid" for n in n_terms],
    y=sig_adds,
    name='Sigmoid Adds',
))

fig.add_trace(go.Bar(
    x=[f"{n}<br>Sigmoid" for n in n_terms],
    y=sig_mults,
    name='Sigmoid Multiplies',
))

# Tanh
fig.add_trace(go.Bar(
    x=[f"{n}<br>Tanh" for n in n_terms],
    y=tanh_adds,
    name='Tanh Adds',
))

fig.add_trace(go.Bar(
    x=[f"{n}<br>Tanh" for n in n_terms],
    y=tanh_mults,
    name='Tanh Multiplies',
))

# Levels (secondary axis style)
fig.add_trace(go.Scatter(
    x=[f"{n}<br>Sigmoid" for n in n_terms],
    y=sig_levels,
    mode='lines+markers',
    name='Sigmoid Level',
    yaxis='y2'
))

fig.add_trace(go.Scatter(
    x=[f"{n}<br>Tanh" for n in n_terms],
    y=tanh_levels,
    mode='lines+markers',
    name='Tanh Level',
    yaxis='y2'
))

fig.update_layout(
    # title='Sigmoid vs Tanh Operation Counts and Levels',
    xaxis_title='Number of Terms / Function',
    yaxis=dict(
        title='Operation Count'
    ),
    yaxis2=dict(
        title='Level',
        overlaying='y',
        side='right'
    ),
    barmode='group',
    bargap=0.35,
    bargroupgap=0.08,
    template='plotly_white',
    width=1200,
    height=600,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    )
)

fig.show()